# Native AHR — environment and restore requirements

Purpose: record installed third-party dependency versions and GPU/runtime facts needed to restore the experiment on another machine. Inputs: current environment, original GNU Go identity definitions, persistent workspace. Outputs: `reports/ahr_native_v4/environment.json`, which is included in the verified local final bundle. Expected runtime: under 15 seconds. This notebook performs no training and writes no model checkpoint.

Resume: Run All in a fresh `ahr-go` kernel. The report describes the inspected environment; it does not claim a matching GPU remains available after shutdown. All authored project executable logic remains in notebooks.

Restore: extract the local final bundle to a workspace, recreate `.venv` and the `ahr-go` kernel using notebook 00 and the recorded package versions, install GNU Go matching the recorded binary SHA, then use notebook 04's unchanged-source/config resume. The final backup validates archived scientific state and inputs as well as CPU inference; runtime dependencies need reinstalling on a new instance. Off-instance transfer awaits the user's authorized Hugging Face destination.


In [ ]:
from pathlib import Path
import nbformat,sys,platform,datetime as dt
from importlib import metadata
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT: raise RuntimeError('Run inside AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_native_v4/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'shared:ahr-export','exec'),globals())
v4_load_upstream(PROJECT,globals())
reference=arena_gnugo_player(PROJECT,'environment-reference',level=1,seed=2026100107,command_timeout_s=3.)
packages=sorted({dist.metadata['Name']:dist.version for dist in metadata.distributions() if dist.metadata.get('Name')}.items())
report={'schema':V4_SCHEMA,'captured_at_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
    'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'torch_cuda':torch.version.cuda,
    'gpu_1':torch.cuda.get_device_name(1) if torch.cuda.device_count()>1 else None,
    'gnu_identity':reference.provenance,'packages':[{'name':name,'version':version} for name,version in packages],
    'workspace_storage':'Persistent boot disk /dev/sda1; instance-deletion policy remains unverified.',
    'off_instance_backup':'DEFERRED_BY_USER_PENDING_HUGGING_FACE_DESTINATION',
    'restore':'Reinstall recorded Python dependencies and pinned GNU binary; restore bundled inputs and scientific/controller checkpoints; resume unchanged notebook04 protocol.'}
v4_atomic_json(v4_report_path(PROJECT,'environment.json'),report)
print({'status':'RECORDED','python':platform.python_version(),'torch':report['torch'],'gpu_1':report['gpu_1'],
       'packages':len(packages),'report':'reports/ahr_native_v4/environment.json'})
